Keypoint Combining – MediaPipe World & MoveNet

Uses `pose_world_landmarks` (3D metric coords, x and y only) instead of normalized image coords.
Output saved to separate folders (`keypoints_combined_world`) to allow comparison with the original.

Shared joints (nose, left_ear, shoulders, elbows, wrists, knees, ankles):
1. MediaPipe world visibility ≥ 0.6 → use MediaPipe world
2. MoveNet confidence ≥ 0.6 → use MoveNet
3. Else → missing (NaN), rescued by GPR as last resort

Hip joints (`left_hip`, `right_hip`):
1. MoveNet confidence ≥ 0.6 → use MoveNet
2. Else if MediaPipe world visibility ≥ 0.6 → use MediaPipe world
3. Else → missing (NaN), rescued by GPR as last resort

MediaPipeonly joints (heels, foot indices):
1. MediaPipe world visibility ≥ 0.6 → use MediaPipe world
2. Else → missing (NaN), rescued by GPR as last resort

Keypoint outputs in this notebook use `mediapipe_world` for MediaPipe joints and MoveNet for selected replacements.
`mediapipe_norm` is not used in the saved keypoint CSVs.

Video overlay is only a visualization helper. It uses `mediapipe_norm` pixelspace coordinates for drawing,
because world coords (meters) cannot be mapped directly onto the video frame.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


IN_DIR           = Path('../data/processed/keypoints_interpolated_boundary')
DOWN_IN_DIR      = Path('../data/processed/keypoints_interpolated_down_boundary')


CROPPED_DIR      = Path('../data/processed/keypoints_cropped')
DOWN_CROPPED_DIR = Path('../data/processed/keypoints_cropped_down')

VIDEO_DIR        = Path('../data/original/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt')


OUT_CSV          = Path('../data/processed/keypoints_combined_world')
DOWN_OUT_CSV     = Path('../data/processed/keypoints_combined_world_down')
OUT_VIDEO        = Path('../data/processed/keypoints_combined_world_videos')
DOWN_OUT_VIDEO   = Path('../data/processed/keypoints_combined_world_videos_down')

FPS            = 50
THRESHOLD_HIGH = 0.6
THRESHOLD_LOW  = 0.3
HIP_CONF_THRESHOLD   = 0.2
SMOOTH_MEDIAN_WINDOW = 5
SMOOTH_MEAN_WINDOW   = 9

time_dict = {
    '028': (9.90,  11.77), '030': (5.80,  7.63),
    '045': (6.54,   7.98), '047': (7.03,  8.91),
    '059': (6.65,   9.11), '061': (6.00,  9.00),
    '074': (6.07,   7.27), '076': (5.00,  8.00),
    '091': (6.39,   7.61), '093': (5.86,  7.67),
}

stand_to_sit_time_dict = {
    '028': (11.87, 12.87), '030': (8.07,  11.56),
    '045': (7.94,   9.26), '047': (9.20,  12.30),
    '059': (9.11,  11.06), '061': (10.57, 13.41),
    '074': (7.27,   8.20), '076': (8.27,  10.84),
    '091': (7.56,   8.18), '093': (8.50,  11.19),
}

JOINTS = [
    'nose',
    'left_ear',
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

MEDIAPIPE_EXTRA_JOINTS = [
    'left_heel',       'right_heel',
    'left_foot_index', 'right_foot_index',
]

ALL_JOINTS = JOINTS + MEDIAPIPE_EXTRA_JOINTS

OUT_CSV.mkdir(parents=True, exist_ok=True)
DOWN_OUT_CSV.mkdir(parents=True, exist_ok=True)
OUT_VIDEO.mkdir(parents=True, exist_ok=True)
DOWN_OUT_VIDEO.mkdir(parents=True, exist_ok=True)

print('Output dirs ready')
print(f'  {OUT_CSV}')
print(f'  {DOWN_OUT_CSV}')
print(f'  {OUT_VIDEO}')
print(f'  {DOWN_OUT_VIDEO}')


Output dirs ready
  ../data/processed/keypoints_combined_world
  ../data/processed/keypoints_combined_world_down
  ../data/processed/keypoints_combined_world_videos
  ../data/processed/keypoints_combined_world_videos_down


Combining function


In [2]:
def smooth_keypoint(values, confidence):
    series = values.astype(float).copy()
    series[confidence.fillna(0).astype(float) < HIP_CONF_THRESHOLD] = np.nan
    series = series.interpolate(limit_direction='both')
    series = series.rolling(SMOOTH_MEDIAN_WINDOW, center=True, min_periods=1).median()
    series = series.rolling(SMOOTH_MEAN_WINDOW, center=True, min_periods=1).mean()
    return series


def combine_keypoints(df_mp, df_mn):
    combined = df_mp.copy()

    hip_joints = {'left_hip', 'right_hip'}
    smoothed_hips = {
        'left_hip': {
            'x': smooth_keypoint(df_mn['left_hip_x'], df_mn['left_hip_confidence']),
            'y': smooth_keypoint(df_mn['left_hip_y'], df_mn['left_hip_confidence']),
        },
        'right_hip': {
            'x': smooth_keypoint(df_mn['right_hip_x'], df_mn['right_hip_confidence']),
            'y': smooth_keypoint(df_mn['right_hip_y'], df_mn['right_hip_confidence']),
        },
    }

    for joint in JOINTS:
        mp_vis  = df_mp[f'{joint}_visibility'].values
        mp_x    = df_mp[f'{joint}_x'].values

        mn_conf   = df_mn[f'{joint}_confidence'].values
        mn_interp = df_mn[f'{joint}_interpolated'].values.astype(bool) if f'{joint}_interpolated' in df_mn.columns else np.zeros(len(df_mn), dtype=bool)
        mn_x      = df_mn[f'{joint}_x'].values

        mp_usable = (mp_vis >= THRESHOLD_HIGH) & ~np.isnan(mp_x)
        mn_usable = (mn_conf >= THRESHOLD_HIGH) | (mn_interp & ~np.isnan(mn_x))

        if joint in hip_joints:
            use_movenet = mn_usable
            use_mediapipe = ~mn_usable & mp_usable
            still_missing = ~mn_usable & ~mp_usable
        else:
            use_movenet = ~mp_usable & mn_usable
            use_mediapipe = mp_usable
            still_missing = ~mp_usable & ~mn_usable

        for coord in ['x', 'y']:
            if joint in hip_joints:
                combined.loc[use_movenet, f'{joint}_{coord}'] = smoothed_hips[joint][coord].loc[use_movenet].values
            else:
                combined.loc[use_movenet, f'{joint}_{coord}'] = df_mn.loc[use_movenet, f'{joint}_{coord}'].values

        combined[f'{joint}_source'] = 'mediapipe'
        combined.loc[use_mediapipe, f'{joint}_source'] = 'mediapipe'
        combined.loc[use_movenet,   f'{joint}_source'] = 'movenet'
        combined.loc[still_missing, f'{joint}_source'] = 'missing'
        combined.loc[still_missing, f'{joint}_x'] = np.nan
        combined.loc[still_missing, f'{joint}_y'] = np.nan

    for joint in MEDIAPIPE_EXTRA_JOINTS:
        mp_vis    = df_mp[f'{joint}_visibility'].values
        mp_x      = df_mp[f'{joint}_x'].values
        mp_usable = (mp_vis >= THRESHOLD_HIGH) & ~np.isnan(mp_x)

        combined[f'{joint}_source'] = 'mediapipe'
        combined.loc[~mp_usable, f'{joint}_source'] = 'missing'
        combined.loc[~mp_usable, f'{joint}_x'] = np.nan
        combined.loc[~mp_usable, f'{joint}_y'] = np.nan

    return combined


Combine and save

MediaPipe world loaded from `keypoints_cropped/mediapipe_world/` (no boundary interpolation needed  minimal effect for MediaPipe).
MoveNet loaded from `keypoints_interpolated_boundary/movenet/` (boundary interpolation important for MoveNet).


In [3]:
for cropped_dir, in_dir, out_csv, label in [
    (CROPPED_DIR,      IN_DIR,      OUT_CSV,      'sit-to-stand'),
    (DOWN_CROPPED_DIR, DOWN_IN_DIR, DOWN_OUT_CSV, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    for mp_path in sorted((cropped_dir / 'mediapipe_world').glob('*.csv')):
        video_id = mp_path.stem.replace('_mediapipe_world', '')
        out_path = out_csv / f'{video_id}.csv'

        if out_path.exists():
            print(f'{video_id}: already exists, skipping')
            continue

        mn_path = in_dir / 'movenet' / f'{video_id}_movenet.csv'
        if not mn_path.exists():
            print(f'  no MoveNet match for {video_id}, skipping')
            continue

        df_mp = pd.read_csv(mp_path)
        df_mn = pd.read_csv(mn_path)

        n           = min(len(df_mp), len(df_mn))
        df_combined = combine_keypoints(df_mp.iloc[:n].reset_index(drop=True),
                                        df_mn.iloc[:n].reset_index(drop=True))

        df_combined.to_csv(out_path, index=False)
        print(f'{video_id}: {n} frames saved')



=== sit-to-stand ===
DJI_20250425092743_0028_D: already exists, skipping
DJI_20250425093100_0030_D: already exists, skipping
DJI_20250425104507_0045_D: already exists, skipping
DJI_20250425104804_0047_D: already exists, skipping
DJI_20250425112502_0059_D: already exists, skipping
DJI_20250425112749_0061_D: already exists, skipping
DJI_20250425120835_0074_D: already exists, skipping
DJI_20250425121226_0076_D: already exists, skipping
DJI_20250425125202_0091_D: already exists, skipping
DJI_20250425125448_0093_D: already exists, skipping

=== stand-to-sit ===
DJI_20250425092743_0028_D: already exists, skipping
DJI_20250425093100_0030_D: already exists, skipping
DJI_20250425104507_0045_D: already exists, skipping
DJI_20250425104804_0047_D: already exists, skipping
DJI_20250425112502_0059_D: already exists, skipping
DJI_20250425112749_0061_D: already exists, skipping
DJI_20250425120835_0074_D: already exists, skipping
DJI_20250425121226_0076_D: already exists, skipping
DJI_20250425125202_0

Missing joints before GPR


In [4]:
missing_before_all = {}

for out_csv, label in [(OUT_CSV, 'sit-to-stand'), (DOWN_OUT_CSV, 'stand-to-sit')]:
    before_rows = []
    for csv_path in sorted(out_csv.glob('*.csv')):
        df = pd.read_csv(csv_path)
        for joint in ALL_JOINTS:
            src_col = f'{joint}_source'
            if src_col not in df.columns:
                continue
            counts = df[src_col].value_counts()
            before_rows.append({
                'video':          csv_path.stem,
                'joint':          joint,
                'missing_before': counts.get('missing', 0),
                'total_frames':   len(df),
            })

    missing_before = pd.DataFrame(before_rows, columns=['video', 'joint', 'missing_before', 'total_frames'])
    missing_before_all[label] = missing_before
    total_before = missing_before['missing_before'].sum()

    print(f'\n=== {label} ===')
    print(f'Missing before GPR: {total_before} frames')
    only_missing = missing_before[missing_before['missing_before'] > 0][['video', 'joint', 'missing_before', 'total_frames']]
    print(only_missing.to_string(index=False))



=== sit-to-stand ===
Missing before GPR: 0 frames
Empty DataFrame
Columns: [video, joint, missing_before, total_frames]
Index: []

=== stand-to-sit ===
Missing before GPR: 0 frames
Empty DataFrame
Columns: [video, joint, missing_before, total_frames]
Index: []


GPR Rescue, conf 0.30.6

For missing joints:
1. Copy coords from `keypoints_cropped/mediapipe_world` if visibility 0.30.6 → flag `low_confidence`
2. Fit GP per joint: highconf frames as anchors, lowconf as noisy observations
3. Predict `low_confidence` + short missing gaps (≤ MAX_GAP_FILL) → source = `gpr`


In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

NOISE_HIGH   = 1e-3
NOISE_LOW    = 0.10
MAX_GAP_FILL = 30


def make_kernel():
    return (ConstantKernel(1.0)
            * RBF(length_scale=5.0, length_scale_bounds=(1.0, 50.0))
            + WhiteKernel(noise_level=NOISE_HIGH, noise_level_bounds=(1e-5, 0.5)))


def find_short_gaps(src_arr, max_gap):
    missing = src_arr == 'missing'
    result  = np.zeros(len(src_arr), dtype=bool)
    i = 0
    while i < len(src_arr):
        if missing[i]:
            j = i
            while j < len(src_arr) and missing[j]:
                j += 1
            if (j - i) <= max_gap:
                result[i:j] = True
            i = j
        else:
            i += 1
    return result


def refine_joint_gpr(df, joint):
    src_col = f'{joint}_source'
    x_col   = f'{joint}_x'
    y_col   = f'{joint}_y'

    src = df[src_col].values
    t   = np.arange(len(df), dtype=float)

    mask_high = np.isin(src, ['mediapipe', 'movenet'])
    mask_low  = src == 'low_confidence'
    mask_fill = find_short_gaps(src, MAX_GAP_FILL)

    if mask_high.sum() < 3 or (mask_low | mask_fill).sum() == 0:
        return df

    mask_train = mask_high | mask_low
    t_train    = t[mask_train]
    alpha      = np.where(mask_high[mask_train], NOISE_HIGH, NOISE_LOW)

    for coord_col in [x_col, y_col]:
        y_train = df[coord_col].values.astype(float)[mask_train]
        if np.any(np.isnan(y_train)):
            continue

        gpr = GaussianProcessRegressor(
            kernel=make_kernel(), alpha=alpha,
            n_restarts_optimizer=3, normalize_y=True, random_state=42,
        )
        gpr.fit(t_train.reshape(-1, 1), y_train)

        if mask_low.sum() > 0:
            y_pred, _ = gpr.predict(t[mask_low].reshape(-1, 1), return_std=True)
            df.loc[mask_low, coord_col] = y_pred

        if mask_fill.sum() > 0:
            y_pred_fill, _ = gpr.predict(t[mask_fill].reshape(-1, 1), return_std=True)
            df.loc[mask_fill, coord_col] = y_pred_fill

    if mask_low.sum() > 0:
        df.loc[mask_low,  src_col] = 'gpr'
    if mask_fill.sum() > 0:
        df.loc[mask_fill, src_col] = 'gpr'

    return df


In [6]:
for out_csv, cropped_dir, label in [
    (OUT_CSV,      CROPPED_DIR,      'sit-to-stand'),
    (DOWN_OUT_CSV, DOWN_CROPPED_DIR, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    cr_mn_dir = cropped_dir / 'movenet'
    cr_mp_dir = cropped_dir / 'mediapipe_world'

    for csv_path in sorted(out_csv.glob('*.csv')):
        df       = pd.read_csv(csv_path)
        video_id = csv_path.stem

        mn_cr_path = cr_mn_dir / f'{video_id}_movenet.csv'
        mp_cr_path = cr_mp_dir / f'{video_id}_mediapipe_world.csv'
        if not mn_cr_path.exists() or not mp_cr_path.exists():
            print(f'{video_id}: cropped files not found, skipping GPR')
            continue

        df_mn_cr = pd.read_csv(mn_cr_path)
        df_mp_cr = pd.read_csv(mp_cr_path)
        n        = min(len(df), len(df_mn_cr), len(df_mp_cr))
        df       = df.iloc[:n].reset_index(drop=True)
        df_mn_cr = df_mn_cr.iloc[:n].reset_index(drop=True)
        df_mp_cr = df_mp_cr.iloc[:n].reset_index(drop=True)

        rescued = []
        for joint in ALL_JOINTS:
            src_col = f'{joint}_source'
            if src_col not in df.columns:
                continue

            missing_mask = df[src_col] == 'missing'
            if missing_mask.sum() == 0:
                continue

            mp_vis = df_mp_cr[f'{joint}_visibility'].values
            mp_low = (mp_vis >= THRESHOLD_LOW) & (mp_vis < THRESHOLD_HIGH)
            use_mp_low = missing_mask & mp_low

            if joint in JOINTS:
                mn_conf    = df_mn_cr[f'{joint}_confidence'].values
                mn_low     = (mn_conf >= THRESHOLD_LOW) & (mn_conf < THRESHOLD_HIGH)
                use_mn_low = missing_mask & ~mp_low & mn_low
            else:
                use_mn_low = np.zeros(len(df), dtype=bool)

            if use_mp_low.any():
                for coord in ['x', 'y']:
                    df.loc[use_mp_low, f'{joint}_{coord}'] = df_mp_cr.loc[use_mp_low, f'{joint}_{coord}'].values
                df.loc[use_mp_low, src_col] = 'low_confidence'

            if use_mn_low.any():
                for coord in ['x', 'y']:
                    df.loc[use_mn_low, f'{joint}_{coord}'] = df_mn_cr.loc[use_mn_low, f'{joint}_{coord}'].values
                df.loc[use_mn_low, src_col] = 'low_confidence'

            df = refine_joint_gpr(df, joint)

            n_gpr = (df[src_col] == 'gpr').sum()
            if n_gpr > 0:
                rescued.append(f'{joint}:{n_gpr}')

        df.to_csv(csv_path, index=False)
        if rescued:
            print(f'{video_id}: GPR rescued {", ".join(rescued)}')
        else:
            print(f'{video_id}: no GPR rescues')



=== sit-to-stand ===
DJI_20250425092743_0028_D: no GPR rescues
DJI_20250425093100_0030_D: no GPR rescues
DJI_20250425104507_0045_D: no GPR rescues
DJI_20250425104804_0047_D: no GPR rescues
DJI_20250425112502_0059_D: no GPR rescues
DJI_20250425112749_0061_D: no GPR rescues
DJI_20250425120835_0074_D: no GPR rescues
DJI_20250425121226_0076_D: no GPR rescues
DJI_20250425125202_0091_D: no GPR rescues
DJI_20250425125448_0093_D: no GPR rescues

=== stand-to-sit ===
DJI_20250425092743_0028_D: no GPR rescues
DJI_20250425093100_0030_D: no GPR rescues
DJI_20250425104507_0045_D: no GPR rescues
DJI_20250425104804_0047_D: no GPR rescues
DJI_20250425112502_0059_D: no GPR rescues
DJI_20250425112749_0061_D: no GPR rescues
DJI_20250425120835_0074_D: no GPR rescues
DJI_20250425121226_0076_D: no GPR rescues
DJI_20250425125202_0091_D: no GPR rescues
DJI_20250425125448_0093_D: no GPR rescues


Hybrid world base + smoothed MoveNet hips

This step is now redundant.
The main `keypoints_combined_world*` outputs already use MoveNet hips with smoothing applied directly during combine.


Step 1  Confidence / Visibility Statistics

Mean and std per joint and overall, across all frames and all videos (sittostand + standtosit).


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def collect_confidence(dirs, conf_col_suffix):
    rows = []
    for d in dirs:
        for csv_path in sorted(d.glob('*.csv')):
            df = pd.read_csv(csv_path)
            for joint in ALL_JOINTS:
                col = f'{joint}_{conf_col_suffix}'
                if col not in df.columns:
                    continue
                for v in df[col].dropna().values:
                    rows.append({'joint': joint, 'value': float(v)})
    return pd.DataFrame(rows)


mn_dirs = [CROPPED_DIR / 'movenet', DOWN_CROPPED_DIR / 'movenet']
mp_dirs = [CROPPED_DIR / 'mediapipe_world', DOWN_CROPPED_DIR / 'mediapipe_world']

df_mn      = collect_confidence(mn_dirs, 'confidence')
df_mp_vis  = collect_confidence(mp_dirs, 'visibility')


t_mn = (df_mn.groupby('joint')['value'].agg(Mean='mean', Std='std').round(4).reindex(ALL_JOINTS))
t_mp = (df_mp_vis.groupby('joint')['value'].agg(Mean='mean', Std='std').round(4).reindex(ALL_JOINTS))

print('=== MoveNet confidence per joint ===')
print(t_mn.to_string())
print('\n=== MediaPipe world visibility per joint ===')
print(t_mp.to_string())


overall = pd.DataFrame([
    {'Model': 'MoveNet (confidence)',   'Mean': round(df_mn['value'].mean(), 4),     'Std': round(df_mn['value'].std(), 4)},
    {'Model': 'MediaPipe (visibility)', 'Mean': round(df_mp_vis['value'].mean(), 4), 'Std': round(df_mp_vis['value'].std(), 4)},
])
print('\n=== Overall (Model, Mean, Std) ===')
print(overall.to_string(index=False))


=== MoveNet confidence per joint ===
                    Mean     Std
joint                           
nose              0.7007  0.1052
left_ear          0.7427  0.1112
left_shoulder     0.8026  0.0877
right_shoulder    0.7884  0.0875
left_elbow        0.7215  0.1460
right_elbow       0.7211  0.1245
left_wrist        0.6013  0.1486
right_wrist       0.6041  0.1624
left_hip          0.8129  0.0886
right_hip         0.7433  0.0937
left_knee         0.7069  0.1465
right_knee        0.7932  0.0926
left_ankle        0.8085  0.0894
right_ankle       0.8015  0.0868
left_heel            NaN     NaN
right_heel           NaN     NaN
left_foot_index      NaN     NaN
right_foot_index     NaN     NaN

=== MediaPipe world visibility per joint ===
                    Mean     Std
joint                           
nose              0.9998  0.0004
left_ear          0.9999  0.0003
left_shoulder     1.0000  0.0000
right_shoulder    0.9999  0.0001
left_elbow        0.9536  0.0917
right_elbow       0.5135  

Step 2  Confidence Distribution Histograms

Red dashed = 0.6, orange dashed = 0.3.


In [ ]:
HIST_DIR = Path('../data/processed/keypoints_combined_world_videos')
HIST_DIR.mkdir(parents=True, exist_ok=True)

def plot_histograms(df_all, title, filename):
    ncols = 5
    nrows = -(-len(ALL_JOINTS) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3))
    axes = axes.flatten()
    for ax, joint in zip(axes, ALL_JOINTS):
        vals = df_all[df_all['joint'] == joint]['value'].values
        ax.hist(vals, bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
        ax.axvline(0.6, color='red',    linestyle='--', linewidth=1.2, label='0.6')
        ax.axvline(0.3, color='orange', linestyle='--', linewidth=1.2, label='0.3')
        ax.set_title(joint, fontsize=9)
        ax.set_xlabel('Score', fontsize=8)
        ax.set_ylabel('Frames', fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in axes[len(ALL_JOINTS):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=8)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = HIST_DIR / filename
    plt.savefig(out, dpi=120)
    plt.close()
    print(f'Saved {out}')

plot_histograms(df_mn,     'MoveNet - Confidence per joint',         'hist_movenet_confidence.png')
plot_histograms(df_mp_vis, 'MediaPipe world - Visibility per joint', 'hist_mediapipe_visibility.png')


Step 3  Threshold Selection

For each threshold: % of keypoints below it, and count of gaps > 10 frames.


In [9]:
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

def threshold_report(dirs, model, conf_col_suffix):
    rows = []
    for thr in THRESHOLDS:
        n_total = n_below = n_long_gaps = 0
        for d in dirs:
            for csv_path in sorted(d.glob('*.csv')):
                df = pd.read_csv(csv_path)
                for joint in ALL_JOINTS:
                    col = f'{joint}_{conf_col_suffix}'
                    if col not in df.columns:
                        continue
                    vals = df[col].fillna(0).values
                    n_total += len(vals)
                    below = vals < thr
                    n_below += below.sum()
                    i = 0
                    while i < len(below):
                        if below[i]:
                            j = i
                            while j < len(below) and below[j]:
                                j += 1
                            if (j - i) > 10:
                                n_long_gaps += 1
                            i = j
                        else:
                            i += 1
        rows.append({
            'Model': model,
            'Threshold': thr,
            '% below': round(100 * n_below / max(n_total, 1), 1),
            'Long gaps >10f': n_long_gaps,
        })
    return pd.DataFrame(rows)

all_mn_dirs = [CROPPED_DIR / 'movenet',         DOWN_CROPPED_DIR / 'movenet']
all_mp_dirs = [CROPPED_DIR / 'mediapipe_world', DOWN_CROPPED_DIR / 'mediapipe_world']

report = pd.concat([
    threshold_report(all_mn_dirs, 'MoveNet',           'confidence'),
    threshold_report(all_mp_dirs, 'MediaPipe vis',     'visibility'),
], ignore_index=True)

print(report.to_string(index=False))


        Model  Threshold  % below  Long gaps >10f
      MoveNet        0.3      0.5               1
      MoveNet        0.4      2.2               4
      MoveNet        0.5      6.5              35
      MoveNet        0.6     14.9              74
      MoveNet        0.7     31.7             138
      MoveNet        0.8     61.1             280
MediaPipe vis        0.3      1.8              19
MediaPipe vis        0.4      3.5              35
MediaPipe vis        0.5      6.0              57
MediaPipe vis        0.6      8.5              70
MediaPipe vis        0.7     11.7              88
MediaPipe vis        0.8     17.0             124


Step 4  Gap Filling

Linear interpolation for gaps <= 10 consecutive NaN frames per joint.  
Gaps > 10 frames are left as NaN.


In [ ]:
NORM_OUT_DIR      = Path('../data/processed/keypoints_normalized_world')
NORM_DOWN_OUT_DIR = Path('../data/processed/keypoints_normalized_world_down')
STD_DIR           = Path('../data/processed/keypoints_standardized_world')
STD_DOWN_DIR      = Path('../data/processed/keypoints_standardized_world_down')

for d in [NORM_OUT_DIR, NORM_DOWN_OUT_DIR, STD_DIR, STD_DOWN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MAX_GAP = 10

def fill_short_gaps(df, max_gap=MAX_GAP):
    df = df.copy()
    for joint in ALL_JOINTS:
        for col in [f'{joint}_x', f'{joint}_y']:
            if col not in df.columns:
                continue
            s = df[col].copy()
            nan_mask = s.isna()
            if not nan_mask.any():
                continue
            long_gap = np.zeros(len(s), dtype=bool)
            i = 0
            while i < len(s):
                if nan_mask.iloc[i]:
                    j = i
                    while j < len(s) and nan_mask.iloc[j]:
                        j += 1
                    if (j - i) > max_gap:
                        long_gap[i:j] = True
                    i = j
                else:
                    i += 1
            interpolated = s.interpolate(method='linear', limit=max_gap, limit_direction='both')
            interpolated[long_gap] = np.nan
            df[col] = interpolated
    return df


for in_csv, out_dir, label in [
    (OUT_CSV,      NORM_OUT_DIR,      'sit-to-stand'),
    (DOWN_OUT_CSV, NORM_DOWN_OUT_DIR, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    for csv_path in sorted(in_csv.glob('*.csv')):
        df = pd.read_csv(csv_path)
        nan_before = df[[f'{j}_x' for j in ALL_JOINTS if f'{j}_x' in df.columns]].isna().sum().sum()
        df_filled  = fill_short_gaps(df)
        nan_after  = df_filled[[f'{j}_x' for j in ALL_JOINTS if f'{j}_x' in df.columns]].isna().sum().sum()
        df_filled.to_csv(out_dir / csv_path.name, index=False)
        print(f'{csv_path.stem}: {nan_before} NaN -> {nan_after} NaN (filled {nan_before - nan_after})')


Step 5  Pose Normalization

Perframe normalization:
Origin: shoulder center = (0, 0)
Scale: torso size (Euclidean distance shoulder center to hip center) = 1


In [11]:
NORM_JOINTS = [j for j in ALL_JOINTS if j != 'left_ear']

def normalize_pose(df):
    df = df.copy()
    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2
    hc_x = (df['left_hip_x']      + df['right_hip_x'])      / 2
    hc_y = (df['left_hip_y']      + df['right_hip_y'])      / 2
    torso_size = np.sqrt((sc_x - hc_x)**2 + (sc_y - hc_y)**2)
    invalid = (torso_size == 0) | torso_size.isna()
    for joint in NORM_JOINTS:
        x_col, y_col = f'{joint}_x', f'{joint}_y'
        if x_col not in df.columns:
            continue
        df[x_col] = (df[x_col] - sc_x) / torso_size
        df[y_col] = (df[y_col] - sc_y) / torso_size
        df.loc[invalid, x_col] = np.nan
        df.loc[invalid, y_col] = np.nan
    return df


for out_dir, label in [
    (NORM_OUT_DIR,      'sit-to-stand'),
    (NORM_DOWN_OUT_DIR, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    for csv_path in sorted(out_dir.glob('*.csv')):
        if csv_path.name.startswith('stability'):
            continue
        df      = pd.read_csv(csv_path)
        df_norm = normalize_pose(df)
        df_norm.to_csv(csv_path, index=False)
        print(f'{csv_path.stem}: {len(df_norm)} frames normalized')


csv_path = sorted(p for p in NORM_OUT_DIR.glob('*.csv') if not p.name.startswith('stability'))[0]
df_v = pd.read_csv(csv_path).dropna(subset=['left_shoulder_x'])
sc_x = (df_v['left_shoulder_x'] + df_v['right_shoulder_x']) / 2
sc_y = (df_v['left_shoulder_y'] + df_v['right_shoulder_y']) / 2
print(f'\nVerify {csv_path.stem}:')
print(f'  shoulder center x - mean: {sc_x.mean():.6f}, max abs: {sc_x.abs().max():.6f}')
print(f'  shoulder center y - mean: {sc_y.mean():.6f}, max abs: {sc_y.abs().max():.6f}')



=== sit-to-stand ===
DJI_20250425092743_0028_D: 94 frames normalized
DJI_20250425093100_0030_D: 92 frames normalized
DJI_20250425104507_0045_D: 73 frames normalized
DJI_20250425104804_0047_D: 94 frames normalized
DJI_20250425112502_0059_D: 123 frames normalized
DJI_20250425112749_0061_D: 151 frames normalized
DJI_20250425120835_0074_D: 60 frames normalized
DJI_20250425121226_0076_D: 151 frames normalized
DJI_20250425125202_0091_D: 61 frames normalized
DJI_20250425125448_0093_D: 91 frames normalized

=== stand-to-sit ===
DJI_20250425092743_0028_D: 50 frames normalized
DJI_20250425093100_0030_D: 175 frames normalized
DJI_20250425104507_0045_D: 67 frames normalized
DJI_20250425104804_0047_D: 156 frames normalized
DJI_20250425112502_0059_D: 98 frames normalized
DJI_20250425112749_0061_D: 142 frames normalized
DJI_20250425120835_0074_D: 47 frames normalized
DJI_20250425121226_0076_D: 129 frames normalized
DJI_20250425125202_0091_D: 32 frames normalized
DJI_20250425125448_0093_D: 135 frames

Step 6  Temporal Stability

Frametoframe Euclidean displacement per joint in normalized space. Lower = smoother.


In [12]:
rows_stab = []

for out_dir, label in [(NORM_OUT_DIR, 'sit-to-stand'), (NORM_DOWN_OUT_DIR, 'stand-to-sit')]:
    for csv_path in sorted(out_dir.glob('*.csv')):
        if csv_path.name.startswith('stability'):
            continue
        df = pd.read_csv(csv_path)
        disps = []
        for joint in NORM_JOINTS:
            x = df[f'{joint}_x'].values if f'{joint}_x' in df.columns else np.array([])
            y = df[f'{joint}_y'].values if f'{joint}_y' in df.columns else np.array([])
            if len(x) < 2:
                continue
            d = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
            disps.extend(d[~np.isnan(d)])
        rows_stab.append({
            'Video':             csv_path.stem,
            'Movement':          label,
            'Mean Displacement': round(np.mean(disps), 4),
            'Std Displacement':  round(np.std(disps), 4),
        })

df_stab = pd.DataFrame(rows_stab)
df_stab.to_csv(NORM_OUT_DIR / 'stability_per_video.csv', index=False)
print(df_stab.to_string(index=False))


rows_joint = []
for out_dir, label in [(NORM_OUT_DIR, 'sit-to-stand'), (NORM_DOWN_OUT_DIR, 'stand-to-sit')]:
    for csv_path in sorted(out_dir.glob('*.csv')):
        if csv_path.name.startswith('stability'):
            continue
        df = pd.read_csv(csv_path)
        for joint in NORM_JOINTS:
            x_col, y_col = f'{joint}_x', f'{joint}_y'
            if x_col not in df.columns:
                continue
            d = np.sqrt(np.diff(df[x_col].values)**2 + np.diff(df[y_col].values)**2)
            d = d[~np.isnan(d)]
            if len(d) == 0:
                continue
            rows_joint.append({
                'Video': csv_path.stem, 'Movement': label, 'Joint': joint,
                'Mean Displacement': round(np.mean(d), 4),
                'Std Displacement':  round(np.std(d), 4),
            })

df_joint = pd.DataFrame(rows_joint)
df_joint.to_csv(NORM_OUT_DIR / 'stability_per_joint.csv', index=False)
print(f'\nSaved stability_per_joint.csv ({len(df_joint)} rows)')


                    Video     Movement  Mean Displacement  Std Displacement
DJI_20250425092743_0028_D sit-to-stand             0.0101            0.0381
DJI_20250425093100_0030_D sit-to-stand             0.0106            0.0492
DJI_20250425104507_0045_D sit-to-stand             0.0089            0.0217
DJI_20250425104804_0047_D sit-to-stand             0.0073            0.0324
DJI_20250425112502_0059_D sit-to-stand             0.0090            0.0465
DJI_20250425112749_0061_D sit-to-stand             0.0100            0.0519
DJI_20250425120835_0074_D sit-to-stand             0.0166            0.0581
DJI_20250425121226_0076_D sit-to-stand             0.0045            0.0207
DJI_20250425125202_0091_D sit-to-stand             0.0105            0.0291
DJI_20250425125448_0093_D sit-to-stand             0.0059            0.0347
DJI_20250425092743_0028_D stand-to-sit             0.0133            0.0429
DJI_20250425093100_0030_D stand-to-sit             0.0043            0.0282
DJI_20250425

Step 7  Standardization (StandardScaler)


In [13]:
from sklearn.preprocessing import StandardScaler

COORD_COLS = [f'{joint}_{c}' for joint in NORM_JOINTS for c in ['x', 'y']
              if f'{joint}_x' in pd.read_csv(sorted(NORM_OUT_DIR.glob('*.csv'))[0]).columns]

for norm_dir, std_dir, label in [
    (NORM_OUT_DIR,      STD_DIR,      'sit-to-stand'),
    (NORM_DOWN_OUT_DIR, STD_DOWN_DIR, 'stand-to-sit'),
]:
    std_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {label} ===')

    all_frames = pd.concat(
        [pd.read_csv(f)[COORD_COLS] for f in sorted(norm_dir.glob('*.csv'))
         if not f.name.startswith('stability')],
        ignore_index=True
    )

    scaler = StandardScaler()
    scaler.fit(all_frames.dropna())

    for csv_path in sorted(norm_dir.glob('*.csv')):
        if csv_path.name.startswith('stability'):
            continue
        df    = pd.read_csv(csv_path)
        valid = df[COORD_COLS].notna().all(axis=1)
        if valid.any():
            df.loc[valid, COORD_COLS] = scaler.transform(df.loc[valid, COORD_COLS])
        df.to_csv(std_dir / csv_path.name, index=False)
        print(f'{csv_path.name} -> standardized')



=== sit-to-stand ===
DJI_20250425092743_0028_D.csv -> standardized
DJI_20250425093100_0030_D.csv -> standardized
DJI_20250425104507_0045_D.csv -> standardized
DJI_20250425104804_0047_D.csv -> standardized
DJI_20250425112502_0059_D.csv -> standardized
DJI_20250425112749_0061_D.csv -> standardized
DJI_20250425120835_0074_D.csv -> standardized
DJI_20250425121226_0076_D.csv -> standardized
DJI_20250425125202_0091_D.csv -> standardized
DJI_20250425125448_0093_D.csv -> standardized

=== stand-to-sit ===
DJI_20250425092743_0028_D.csv -> standardized
DJI_20250425093100_0030_D.csv -> standardized
DJI_20250425104507_0045_D.csv -> standardized
DJI_20250425104804_0047_D.csv -> standardized
DJI_20250425112502_0059_D.csv -> standardized
DJI_20250425112749_0061_D.csv -> standardized
DJI_20250425120835_0074_D.csv -> standardized
DJI_20250425121226_0076_D.csv -> standardized
DJI_20250425125202_0091_D.csv -> standardized
DJI_20250425125448_0093_D.csv -> standardized


In [14]:
print('Skipped: MoveNet hip smoothing is now applied directly in keypoints_combined_world outputs.')


Skipped: MoveNet hip smoothing is now applied directly in keypoints_combined_world outputs.


Videos

Blue   = MediaPipe world (world x,y mapped to pixels via hip anchor + shoulder scale)
Green  = MoveNet (imagenormalized 01, used directly)
Orange = GPR rescued

World coords (meters, origin at hip center) are mapped to pixels per frame:
Anchor: hip midpoint position from `mediapipe_norm` (stable pixel reference)
Scale: shouldertoshoulder distance in pixels / in meters


In [ ]:
import cv2
import imageio
import subprocess

WORLD_CONNECTIONS = [
    ('left_shoulder', 'right_shoulder'),
    ('left_shoulder', 'left_elbow'),
    ('left_elbow', 'left_wrist'),
    ('right_shoulder', 'right_elbow'),
    ('right_elbow', 'right_wrist'),
    ('left_shoulder', 'left_hip'),
    ('right_shoulder', 'right_hip'),
    ('left_hip', 'right_hip'),
    ('left_hip', 'left_knee'),
    ('left_knee', 'left_ankle'),
    ('right_hip', 'right_knee'),
    ('right_knee', 'right_ankle'),
    ('left_ankle', 'left_heel'),
    ('left_heel', 'left_foot_index'),
    ('right_ankle', 'right_heel'),
    ('right_heel', 'right_foot_index'),
]

JOINT_COLORS = {
    'mediapipe': (255, 80,  80),
    'movenet':   (80,  200, 80),
    'gpr':       (255, 165, 0),
    'missing':   (160, 160, 160),
}


def get_video_rotation(video_path):
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'stream_side_data=rotation:stream_tags=rotate',
        '-of', 'default=noprint_wrappers=1:nokey=0',
        str(video_path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    except Exception:
        return 0
    rotation = 0
    for line in result.stdout.splitlines():
        if line.startswith('rotation='):
            try:
                rotation = int(float(line.split('=', 1)[1]))
            except ValueError:
                rotation = 0
    return rotation


def apply_video_rotation(frame, rotation):
    rotation = rotation % 360
    if rotation == 180:
        return cv2.rotate(frame, cv2.ROTATE_180)
    if rotation == 90:
        return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    if rotation == 270:
        return cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return frame


def compute_world_transform(row, width, height):
    lhx = row.get('left_hip_x')  if row.get('left_hip_source')  == 'movenet' else np.nan
    lhy = row.get('left_hip_y')  if row.get('left_hip_source')  == 'movenet' else np.nan
    rhx = row.get('right_hip_x') if row.get('right_hip_source') == 'movenet' else np.nan
    rhy = row.get('right_hip_y') if row.get('right_hip_source') == 'movenet' else np.nan

    if np.isnan(lhx) or np.isnan(rhx):
        movenet_xs = [row.get(f'{j}_x') for j in ALL_JOINTS
                      if row.get(f'{j}_source') == 'movenet' and not pd.isna(row.get(f'{j}_x'))]
        movenet_ys = [row.get(f'{j}_y') for j in ALL_JOINTS
                      if row.get(f'{j}_source') == 'movenet' and not pd.isna(row.get(f'{j}_y'))]
        lhx = rhx = float(np.nanmedian(movenet_xs)) if movenet_xs else 0.5
        lhy = rhy = float(np.nanmedian(movenet_ys)) if movenet_ys else 0.5

    hip_px = np.nanmean([float(lhx), float(rhx)]) * width
    hip_py = np.nanmean([float(lhy), float(rhy)]) * height

    lsxw = row.get('left_shoulder_x');  rsxw = row.get('right_shoulder_x')
    lsyw = row.get('left_shoulder_y');  rsyw = row.get('right_shoulder_y')
    dist_w = np.sqrt((float(lsxw) - float(rsxw))**2 + (float(lsyw) - float(rsyw))**2)             if not any(pd.isna(v) for v in [lsxw, rsxw, lsyw, rsyw]) else np.nan

    if np.isnan(dist_w) or dist_w < 1e-6:
        scale = height * 0.45
    else:
        scale = height * 0.45 / 1.0

        lkx_mn = row.get('left_knee_x')  if row.get('left_knee_source')  == 'movenet' else np.nan
        lky_mn = row.get('left_knee_y')  if row.get('left_knee_source')  == 'movenet' else np.nan
        lhx_mn = float(lhx); lhy_mn = float(lhy)
        lkxw   = row.get('left_knee_x')  if row.get('left_knee_source')  != 'movenet' else np.nan
        lkyw   = row.get('left_knee_y')  if row.get('left_knee_source')  != 'movenet' else np.nan
        lhxw   = row.get('left_hip_x')   if row.get('left_hip_source')   != 'movenet' else np.nan
        lhyw   = row.get('left_hip_y')   if row.get('left_hip_source')   != 'movenet' else np.nan

        if not any(pd.isna(v) for v in [lkx_mn, lky_mn, lkxw, lkyw, lhxw, lhyw]):
            femur_px    = np.sqrt((lhx_mn*width  - lkx_mn*width )**2 + (lhy_mn*height - lky_mn*height)**2)
            femur_world = np.sqrt((float(lhxw) - float(lkxw))**2 + (float(lhyw) - float(lkyw))**2)
            if femur_world > 1e-6:
                scale = femur_px / femur_world

    return hip_px, hip_py, scale


def draw_frame(frame_rgb, row):
    h, w, _ = frame_rgb.shape
    img = frame_rgb.copy()
    hip_px, hip_py, scale = compute_world_transform(row, w, h)

    pts = {}
    for joint in ALL_JOINTS:
        src = row.get(f'{joint}_source', 'mediapipe')
        x   = row.get(f'{joint}_x')
        y   = row.get(f'{joint}_y')
        if pd.isna(x) or pd.isna(y):
            continue
        if src == 'movenet':
            px, py = int(float(x) * w), int(float(y) * h)
        else:
            px = int(hip_px + float(x) * scale)
            py = int(hip_py - float(y) * scale)
        pts[joint] = (px, py, src)

    for a, b in WORLD_CONNECTIONS:
        if a not in pts or b not in pts:
            continue
        cv2.line(img, pts[a][:2], pts[b][:2], (180, 180, 180), 2, cv2.LINE_AA)

    for joint, (px, py, src) in pts.items():
        color = JOINT_COLORS.get(src, JOINT_COLORS['mediapipe'])
        cv2.circle(img, (px, py), 6, color, -1, cv2.LINE_AA)

    return img


for out_csv, out_video, td, label in [
    (OUT_CSV,      OUT_VIDEO,      time_dict,             'sit-to-stand'),
    (DOWN_OUT_CSV, DOWN_OUT_VIDEO, stand_to_sit_time_dict,'stand-to-sit'),
]:
    out_video.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {label} ===')
    for csv_path in sorted(out_csv.glob('*.csv')):
        match = re.search(r'_0(\d{3})_D', csv_path.stem)
        if not match:
            continue
        vid_id    = match.group(1)
        vid_files = list(VIDEO_DIR.glob(f'*_0{vid_id}_D.MP4'))
        if not vid_files:
            print(f'Video not found: {vid_id}')
            continue

        df          = pd.read_csv(csv_path)
        n           = len(df)
        out_path    = out_video / f'{csv_path.stem}.mp4'
        start_frame = int(td[vid_id][0] * FPS)
        video_path  = vid_files[0]
        rotation    = get_video_rotation(video_path)
        cap         = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f'Could not open: {video_path.name}')
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        with imageio.get_writer(str(out_path), fps=FPS) as writer:
            for i in range(n):
                ok, frame = cap.read()
                if not ok:
                    print(f'{vid_id}: stopped at frame {start_frame + i}')
                    break
                frame = apply_video_rotation(frame, rotation)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                writer.append_data(draw_frame(frame, df.iloc[i]))

        cap.release()
        print(f'{vid_id}: {n} frames -> {out_path.name} (rotation {rotation})')
